In [1]:
from LLMGeometry.datasets import load_dataset_by_name
from LLMGeometry.in_context_learning import ICL_Template, load_ICL_template, list_ICL_templates_in_json
from LLMGeometry import load_model_and_tokenizer
from LLMGeometry.evaluation import run_on_dataframe
from LLMGeometry.utils import generate_random_samples, save_file_with_incremental_suffix
import torch
import pandas as pd
import pickle
from termcolor import colored
from pathlib import Path
import argparse
import json
import sys
from sklearn.metrics import accuracy_score, f1_score
import random
import numpy as np


/gpfs/share/apps/miniconda3/gpu/4.9.2/lib/python3.8/site-packages/requests/__init__.py:102: RequestsDependencyWarning: urllib3 (1.26.8) or chardet (5.2.0)/charset_normalizer (2.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "


In [3]:
from scipy.optimize import linear_sum_assignment

In [3]:
MODEL_NAME = 'llama3.1_1b_base'
DATASET_NAME = 'claude_multitask'
prefix_type = 'demos'
n_examples = 10
keyword = 'Category'
answer_field = 'emotion_letter'
N_RUNS = 50
root_folder = "temp/ICL_results"

In [3]:
datasets = load_dataset_by_name(DATASET_NAME)
train_df = datasets['train']
test_df = datasets['test']
train_df = train_df[train_df['emotion'].isin(['Joy', 'Anger', 'Fear'])]
train_df = train_df.reset_index(drop=True)
test_df = test_df[test_df['emotion'].isin(['Joy', 'Anger', 'Fear'])]
test_df = test_df.reset_index(drop=True)

In [4]:
len(test_df)

300

In [ ]:
test_df.head()

,text,emotion,topic,intent,emotion_label,topic_label,intent_label,emotion_letter,topic_letter,intent_letter,emotion_shuffled,topic_shuffled,intent_shuffled,is_lowercase,is_uppercase
0,"""GENIUS INVENTS REVOLUTIONARY APP THAT ADDS ZE...",Joy,Technology,Sarcastic,0,0,1,A,A,B,Sadness,Politics,Metaphorical,False,True
1,"AFTER MISSING THE PENALTY KICK, I REALLY DROPP...",Anger,Sports,Idiomatic,2,3,4,C,D,E,Fear,Health,Literal,False,True
2,the reality show contestant's carefully crafte...,Anger,Entertainment,Humorous,2,2,3,C,C,D,Fear,Sports,Idiomatic,True,False
3,The vending machine ate my last dollar and had...,Anger,Health,Humorous,2,4,3,C,E,D,Fear,Technology,Idiomatic,False,False
4,Citizens celebrate the astounding 99% approval...,Joy,Politics,Sarcastic,0,1,1,A,B,B,Sadness,Entertainment,Metaphorical,False,False


In [6]:
train_df

,text,emotion,topic,intent,emotion_label,topic_label,intent_label,emotion_letter,topic_letter,intent_letter,emotion_shuffled,topic_shuffled,intent_shuffled,is_lowercase,is_uppercase
0,The underdog candidate's supporters are over t...,Joy,Politics,Idiomatic,0,1,4,A,B,E,Sadness,Entertainment,Literal,False,False
1,My lucky socks finally paid off when I hit a h...,Joy,Sports,Humorous,0,3,3,A,D,D,Sadness,Health,Idiomatic,False,False
2,The corridors of power will become a labyrinth...,Fear,Politics,Metaphorical,3,1,2,D,B,C,Surprise,Entertainment,Humorous,False,False
3,"Oh, I'm sure the new bill they've proposed wil...",Anger,Politics,Sarcastic,2,1,1,C,B,B,Fear,Entertainment,Metaphorical,False,False
4,YOU'VE DONE A STELLAR JOB ELECTING OFFICIALS W...,Anger,Politics,Sarcastic,2,1,1,C,B,B,Fear,Entertainment,Metaphorical,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,Dr. Smith prescribes a strict diet of kale and...,Anger,Health,Humorous,2,4,3,C,E,D,Fear,Technology,Idiomatic,False,False
296,Isn't it wonderful how transparent and honest ...,Joy,Politics,Sarcastic,0,1,1,A,B,B,Sadness,Entertainment,Metaphorical,False,False
297,THE HUMAN BODY WILL BECOME A TICKING TIME BOMB...,Fear,Health,Metaphorical,3,4,2,D,E,C,Surprise,Technology,Humorous,False,True
298,I'm thrilled about the new AI assistant that's...,Fear,Technology,Sarcastic,3,0,1,D,A,B,Surprise,Politics,Metaphorical,False,False


In [7]:
print(test_df['emotion'].unique())

['Joy', 'Anger', 'Fear']
Categories (5, object): ['Joy', 'Sadness', 'Anger', 'Fear', 'Surprise']


In [8]:
def create_template(train_df, prefix_type, n_examples, keyword, answer_field, dataset_name, shuffle_labels=False, seed=None):
    '''
        Create a template for the ICL prompt.
        
        Parameters:
        ----------
        prefix_type: str
            The type of the prefix. Can be 'raw', 'instruction', 'demos'.
        n_examples: int
            The number of examples to include in the prefix.
        keyword: str
            The keyword to be used in the template.
        answer_field: str
            The name of the answer field.
        dataset_name: str
            The name of the dataset.
        shuffle_labels: bool
            Whether to shuffle the labels.
        seed: int
            The seed for the random number generator.
        
        Returns:
        -------
        prefix: str
            The prefix of the ICL prompt.
        suffix: str
            The suffix of the ICL prompt
    '''
    # ds = load_dataset_by_name(dataset_name)
    # train_df = ds['train']
    category_list = train_df[answer_field].unique()
    # --- Creating the prefix 
    if prefix_type == 'raw':
        prefix = ''
    elif prefix_type=='instruction':
        prefix = 'This is a text classification task. Possible categories are: ' + ', '.join(category_list) + '.\n'
    elif prefix_type == 'demos':
        chosen_indices = generate_random_samples(train_df[answer_field], n_examples, seed=seed)
        chosen_sentences = train_df.loc[chosen_indices,'text']
        chosen_labels = train_df.loc[chosen_indices, answer_field]
        if shuffle_labels:
            chosen_labels = chosen_labels.sample(frac=1).reset_index(drop=True) # Shuffle the labels
        prefix = ''
        for sentence, label in zip(chosen_sentences, chosen_labels):
            prefix += 'Text: ' + sentence + f'\n{keyword}: ' + label + '\n'
    # --- Creating the suffix
    suffix = f'\n{keyword}:'
    return prefix, suffix, chosen_sentences, chosen_labels

In [9]:
prefix, suffix, sentences, labels = create_template(train_df, prefix_type, n_examples, keyword, answer_field, DATASET_NAME, seed=42)

In [10]:
print('prefix: ', prefix)
print('suffix: ', suffix)
print('sentences: ', sentences)
print('labels: ', labels)
print(sentences.iloc[0])

prefix:  Text: As the riots intensified, I cowered in my basement, wondering if democracy had finally met its Waterloo.
Category: D
Text: The concert was a tidal wave of sound, washing over the ecstatic crowd.
Category: A
Text: The opposition's venomous rhetoric spews forth, poisoning the well of civil discourse.
Category: C
Text: he's thrilled to bits about his new diet of nothing but deep-fried foods and sugary drinks.
Category: A
Text: The corrupt senator's lies finally caught up with him when the incriminating evidence surfaced.
Category: C
Text: The Wi-Fi connection was so blazingly fast, I managed to load half a webpage in just under an hour.
Category: C
Text: In the coming election, I'll cast my ballot and watch it soar like a dove, carrying the weight of millions of dreams.
Category: A
Text: You find yourself trapped in the cacophony of fame, its siren song slowly drowning your true voice.
Category: D
Text: Standing at the starting line, you'll feel your heart race before the b

In [11]:
# get unique labels
labels.unique()


array(['D', 'A', 'C'], dtype=object)

In [4]:
model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
vocab = tokenizer.get_vocab()
all_tokens = vocab.values()
all_tokens_str = vocab.keys()

# with open('template_sentence_probs_all.pkl', 'rb') as f:
#     sentence_probs = pickle.load(f)
# with open('template_sentence_logits_all.pkl', 'rb') as f:
#     sentence_logits = pickle.load(f)

meta-llama/Llama-3.2-1B Model and tokenizer loaded


In [21]:
tokenizer.decode(tokenizer.convert_tokens_to_ids('ĠìĬ¤íı¬ì¸ł'))
# tokenizer.decode(tokenizer.convert_tokens_to_ids('ĠMÃ©d'))
# # tokenizer.decode(tokenizer.convert_tokens_to_ids('ĠÑˆHÐμÐ»'))
# tokenizer.decode(tokenizer.convert_tokens_to_ids('ĠbaÅŁarÄ±'))
# # tokenizer.decode(tokenizer.convert_tokens_to_ids('ĠÑ ‘gÐ¾Ð½'))
# # tokenizer.decode(tokenizer.convert_tokens_to_ids('ĠÐ½Ð°ÑˆhÐº'))



' 스포츠'

In [ ]:
for sentence, label in zip(sentences, labels): # N
    print(sentence_logits[sentence].shape)
    # index based on token_str?
    print(sentence_logits[sentence][tokenizer.convert_tokens_to_ids('Ġglaciers')])
    break

torch.Size([128256])
tensor(-1.8203, dtype=torch.bfloat16)


In [ ]:
def optimize_tokens(top_k_tokens, sentences, labels, sentence_logits):
    classes = labels.unique()
    # start with random assignment
    token_assignments = {c: random.choice(top_k_tokens) for c in classes}
    sum_support_ex = {}
    for sentence, label in zip(sentences, labels): # N
        logit = sentence_logits[sentence][tokenizer.convert_tokens_to_ids(token_assignments[label])]
        for c in classes:
            denominator += np.exp(sentence_logits[sentence][tokenizer.convert_tokens_to_ids(token_assignments[c])])
        probs = np.exp(logit) / denominator
        log_prob = np.log(probs)
        sum_support_ex += log_prob
        

In [14]:
def optimize_tokens_with_comments(top_k_tokens, sentences, labels, sentence_logits, max_iterations=100):
    classes = labels.unique()
    
    # Convert to list if it's dict_keys
    if hasattr(top_k_tokens, 'keys'):
        top_k_tokens = list(top_k_tokens)
    
    # Helper function to calculate the objective (sum_support_ex)
    def calculate_objective(token_assignments):
        total_log_prob = 0
        for sentence, label in zip(sentences, labels):
            # Convert token string to token ID
            label_token_id = tokenizer.convert_tokens_to_ids(token_assignments[label])
            logit = sentence_logits[sentence][label_token_id]
            
            # Convert to float32 to avoid BFloat16 issues
            if hasattr(logit, 'float'):
                logit = logit.float()
            
            # Calculate denominator
            denominator = 0
            for c in classes:
                c_token_id = tokenizer.convert_tokens_to_ids(token_assignments[c])
                c_logit = sentence_logits[sentence][c_token_id]
                
                # Convert to float32 to avoid BFloat16 issues
                if hasattr(c_logit, 'float'):
                    c_logit = c_logit.float()
                
                denominator += torch.exp(c_logit)
            
            probs = torch.exp(logit) / denominator
            log_prob = torch.log(probs)
            total_log_prob += log_prob.item()  # Use .item() to extract scalar value
        return total_log_prob
    
    # Initialize with random assignment
    token_assignments = {c: random.choice(top_k_tokens) for c in classes}
    current_objective = calculate_objective(token_assignments)
    
    print(f"Initial objective: {current_objective}")
    
    # Hill climbing optimization
    improved = True
    iteration = 0
    
    while improved and iteration < max_iterations:
        improved = False
        iteration += 1
        
        # # Try changing one token assignment at a time
        # for class_to_change in classes:
        #     current_token = token_assignments[class_to_change]
            
        #     # Try all other possible tokens for this class
        #     for new_token in top_k_tokens:
        #         if new_token == current_token:
        #             continue
                    
        #         # Create new assignment with this token change
        #         new_assignments = token_assignments.copy()
        #         new_assignments[class_to_change] = new_token
                
        #         # Calculate new objective
        #         new_objective = calculate_objective(new_assignments)
                
        #         # If improvement found, update and mark as improved
        #         if new_objective > current_objective:
        #             token_assignments = new_assignments
        #             current_objective = new_objective
        #             improved = True
        #             print(f"Iteration {iteration}: Improved objective to {current_objective} by changing {class_to_change} to {new_token}")
        #             break  # Move to next class after finding improvement
            
        #     if improved:
        #         break  # Restart from first class after any improvement
        # Try changing one token assignment at a time
        for class_to_change in classes:
            current_token = token_assignments[class_to_change]
            
            # Get all candidate tokens (excluding current token)
            candidate_tokens = [token for token in top_k_tokens if token != current_token]
            
            if not candidate_tokens:
                continue
            
            # Convert candidate tokens to token IDs
            candidate_token_ids = [tokenizer.convert_tokens_to_ids(token) for token in candidate_tokens]
            candidate_token_ids = torch.tensor(candidate_token_ids)
            
            # Get token IDs for other classes (unchanged)
            other_class_token_ids = []
            for c in classes:
                if c != class_to_change:
                    other_class_token_ids.append(tokenizer.convert_tokens_to_ids(token_assignments[c]))
            other_class_token_ids = torch.tensor(other_class_token_ids)
            
            # Vectorized calculation for all candidates
            total_log_probs = torch.zeros(len(candidate_tokens))
            
            # for sentence, label in zip(sentences, labels):
            #     sentence_logits_tensor = sentence_logits[sentence]
                
            #     if label == class_to_change:
            #         # This sentence uses the class we're changing
            #         label_logits = sentence_logits_tensor[candidate_token_ids].float()
            #         other_logits = sentence_logits_tensor[other_class_token_ids].float()
                    
            #         # For each candidate, calculate denominator and probability
            #         for i, candidate_logit in enumerate(label_logits):
            #             denominator = torch.exp(candidate_logit) + torch.exp(other_logits).sum()
            #             prob = torch.exp(candidate_logit) / denominator
            #             total_log_probs[i] += torch.log(prob)
            #     else:
            #         # This sentence uses a different class (unchanged)
            #         label_token_id = tokenizer.convert_tokens_to_ids(token_assignments[label])
            #         label_logit = sentence_logits_tensor[label_token_id].float()
                    
            #         # Calculate denominator for each candidate
            #         for i, candidate_logit in enumerate(sentence_logits_tensor[candidate_token_ids].float()):
            #             other_unchanged_logits = sentence_logits_tensor[other_class_token_ids].float()
            #             denominator = torch.exp(candidate_logit) + torch.exp(other_unchanged_logits).sum()
            #             prob = torch.exp(label_logit) / denominator
            #             total_log_probs[i] += torch.log(prob)
            
            # Vectorized calculation for all candidates
            for sentence, label in zip(sentences, labels):
                sentence_logits_tensor = sentence_logits[sentence]
                
                if label == class_to_change:
                    # This sentence uses the class we're changing
                    label_logits = sentence_logits_tensor[candidate_token_ids].float()  # Shape: (num_candidates,)
                    other_logits = sentence_logits_tensor[other_class_token_ids].float()  # Shape: (num_other_classes,)
                    
                    # Vectorized denominator calculation
                    # label_logits.unsqueeze(1) + other_logits.sum() gives us all denominators at once
                    denominators = torch.exp(label_logits) + torch.exp(other_logits).sum()  # Shape: (num_candidates,)
                    probs = torch.exp(label_logits) / denominators
                    total_log_probs += torch.log(probs)
                    
                else:
                    # This sentence uses a different class (unchanged)
                    label_token_id = tokenizer.convert_tokens_to_ids(token_assignments[label])
                    label_logit = sentence_logits_tensor[label_token_id].float()
                    
                    # Vectorized denominator calculation
                    candidate_logits = sentence_logits_tensor[candidate_token_ids].float()  # Shape: (num_candidates,)
                    other_unchanged_logits = sentence_logits_tensor[other_class_token_ids].float()  # Shape: (num_other_classes,)
                    
                    denominators = torch.exp(candidate_logits) + torch.exp(other_unchanged_logits).sum()
                    probs = torch.exp(label_logit) / denominators  # Same numerator for all candidates
                    total_log_probs += torch.log(probs)
            
            # Find the best candidate
            best_idx = torch.argmax(total_log_probs).item()
            best_objective = total_log_probs[best_idx].item()
            best_token = candidate_tokens[best_idx]
            
            # If improvement found, update
            if best_objective > current_objective:
                token_assignments[class_to_change] = best_token
                current_objective = best_objective
                improved = True
                print(f"Iteration {iteration}: Improved objective to {best_objective} by changing {class_to_change} to {best_token}")
                break  # Move to next class after finding improvement
    
    print(f"Final objective after {iteration} iterations: {current_objective}")
    print(f"Final token assignments: {token_assignments}")

    return token_assignments, current_objective

In [14]:
def optimize_tokens(top_k_tokens, sentences, labels, sentence_logits, max_iterations=100, lambda_reg=0.0001):
    classes = labels.unique()
    
    # Convert to list if it's dict_keys
    if hasattr(top_k_tokens, 'keys'):
        top_k_tokens = list(top_k_tokens)
    
    # Helper function to calculate the objective (sum_support_ex)
    def calculate_objective(token_assignments):
        total_log_prob = 0
        for sentence, label in zip(sentences, labels):
            # Convert token string to token ID
            label_token_id = tokenizer.convert_tokens_to_ids(token_assignments[label])
            logit = sentence_logits[sentence][label_token_id]
            
            # Convert to float32 to avoid BFloat16 issues
            if hasattr(logit, 'float'):
                logit = logit.float()
            
            # Calculate denominator
            denominator = 0
            for c in classes:
                c_token_id = tokenizer.convert_tokens_to_ids(token_assignments[c])
                c_logit = sentence_logits[sentence][c_token_id]
                
                # Convert to float32 to avoid BFloat16 issues
                if hasattr(c_logit, 'float'):
                    c_logit = c_logit.float()
                
                denominator += torch.exp(c_logit)
            
            probs = torch.exp(logit) / denominator
            log_prob = torch.log(probs)
            total_log_prob += log_prob.item()  # Use .item() to extract scalar value
                
        # Add regularization term for token IDs
        token_id_sum = 0
        for c in classes:
            token_id = tokenizer.convert_tokens_to_ids(token_assignments[c])
            token_id_sum += token_id
        
        # Combine classification loss with regularization
        # Negative because we want to minimize token IDs (encourage early/frequent tokens)
        return total_log_prob - lambda_reg * token_id_sum
    
    # Initialize with random assignment
    token_assignments = {c: random.choice(top_k_tokens) for c in classes}
    current_objective = calculate_objective(token_assignments)
    
    print(f"Initial objective: {current_objective}")
    
    # Hill climbing optimization
    improved = True
    iteration = 0
    
    while improved and iteration < max_iterations:
        improved = False
        iteration += 1
    
        for class_to_change in classes:
            current_token = token_assignments[class_to_change]
            
            # Get all candidate tokens (excluding current token)
            candidate_tokens = [token for token in top_k_tokens if token != current_token]
            
            if not candidate_tokens:
                continue
            
            # Convert candidate tokens to token IDs
            candidate_token_ids = [tokenizer.convert_tokens_to_ids(token) for token in candidate_tokens]
            candidate_token_ids = torch.tensor(candidate_token_ids)
            
            # Get token IDs for other classes (unchanged)
            other_class_token_ids = []
            for c in classes:
                if c != class_to_change:
                    other_class_token_ids.append(tokenizer.convert_tokens_to_ids(token_assignments[c]))
            other_class_token_ids = torch.tensor(other_class_token_ids)
            other_class_id_sum = sum(tokenizer.convert_tokens_to_ids(token_assignments[c]) 
                        for c in classes if c != class_to_change)
            
            # Vectorized calculation for all candidates
            total_log_probs = torch.zeros(len(candidate_tokens))
            
            # Vectorized calculation for all candidates
            for sentence, label in zip(sentences, labels):
                sentence_logits_tensor = sentence_logits[sentence]
                
                if label == class_to_change:
                    # This sentence uses the class we're changing
                    label_logits = sentence_logits_tensor[candidate_token_ids].float()  # Shape: (num_candidates,)
                    other_logits = sentence_logits_tensor[other_class_token_ids].float()  # Shape: (num_other_classes,)
                    
                    # Vectorized denominator calculation
                    # label_logits.unsqueeze(1) + other_logits.sum() gives us all denominators at once
                    denominators = torch.exp(label_logits) + torch.exp(other_logits).sum()  # Shape: (num_candidates,)
                    probs = torch.exp(label_logits) / denominators
                    total_log_probs += torch.log(probs)
                    
                else:
                    # This sentence uses a different class (unchanged)
                    label_token_id = tokenizer.convert_tokens_to_ids(token_assignments[label])
                    label_logit = sentence_logits_tensor[label_token_id].float()
                    
                    # Vectorized denominator calculation
                    candidate_logits = sentence_logits_tensor[candidate_token_ids].float()  # Shape: (num_candidates,)
                    other_unchanged_logits = sentence_logits_tensor[other_class_token_ids].float()  # Shape: (num_other_classes,)
                    
                    denominators = torch.exp(candidate_logits) + torch.exp(other_unchanged_logits).sum()
                    probs = torch.exp(label_logit) / denominators  # Same numerator for all candidates
                    total_log_probs += torch.log(probs)
            
            token_id_penalties = -lambda_reg * (candidate_token_ids.float() + other_class_id_sum)
            total_log_probs += token_id_penalties
            
            # Find the best candidate
            best_idx = torch.argmax(total_log_probs).item()
            best_objective = total_log_probs[best_idx].item()
            best_token = candidate_tokens[best_idx]
            
            # If improvement found, update
            if best_objective > current_objective:
                token_assignments[class_to_change] = best_token
                current_objective = best_objective
                improved = True
                print(f"Iteration {iteration}: Improved objective to {best_objective} by changing {class_to_change} to {best_token}")
                break  # Move to next class after finding improvement
    
    print(f"Final objective after {iteration} iterations: {current_objective}")
    print(f"Final token assignments: {token_assignments}")

    return token_assignments, current_objective

In [ ]:
def optimize_tokens_with_comments(top_k_tokens, sentences, labels, sentence_logits, max_iterations=100):
    classes = labels.unique()
    
    # Convert to list if it's dict_keys
    if hasattr(top_k_tokens, 'keys'):
        top_k_tokens = list(top_k_tokens)
    
    # Helper function to calculate the objective (sum_support_ex)
    def calculate_objective(token_assignments):
        total_log_prob = 0
        for sentence, label in zip(sentences, labels):
            # Convert token string to token ID
            label_token_id = tokenizer.convert_tokens_to_ids(token_assignments[label])
            logit = sentence_logits[sentence][label_token_id]
            
            # Convert to float32 to avoid BFloat16 issues
            if hasattr(logit, 'float'):
                logit = logit.float()
            
            # Calculate denominator
            denominator = 0
            for c in classes:
                c_token_id = tokenizer.convert_tokens_to_ids(token_assignments[c])
                c_logit = sentence_logits[sentence][c_token_id]
                
                # Convert to float32 to avoid BFloat16 issues
                if hasattr(c_logit, 'float'):
                    c_logit = c_logit.float()
                
                denominator += torch.exp(c_logit)
            
            probs = torch.exp(logit) / denominator
            log_prob = torch.log(probs)
            total_log_prob += log_prob.item()  # Use .item() to extract scalar value
        return total_log_prob
    
    # Initialize with random assignment
    token_assignments = {c: random.choice(top_k_tokens) for c in classes}
    current_objective = calculate_objective(token_assignments)
    
    print(f"Initial objective: {current_objective}")
    
    # Hill climbing optimization
    improved = True
    iteration = 0
    
    while improved and iteration < max_iterations:
        improved = False
        iteration += 1
        
        # # Try changing one token assignment at a time
        # for class_to_change in classes:
        #     current_token = token_assignments[class_to_change]
            
        #     # Try all other possible tokens for this class
        #     for new_token in top_k_tokens:
        #         if new_token == current_token:
        #             continue
                    
        #         # Create new assignment with this token change
        #         new_assignments = token_assignments.copy()
        #         new_assignments[class_to_change] = new_token
                
        #         # Calculate new objective
        #         new_objective = calculate_objective(new_assignments)
                
        #         # If improvement found, update and mark as improved
        #         if new_objective > current_objective:
        #             token_assignments = new_assignments
        #             current_objective = new_objective
        #             improved = True
        #             print(f"Iteration {iteration}: Improved objective to {current_objective} by changing {class_to_change} to {new_token}")
        #             break  # Move to next class after finding improvement
            
        #     if improved:
        #         break  # Restart from first class after any improvement
        # Try changing one token assignment at a time
        for class_to_change in classes:
            current_token = token_assignments[class_to_change]
            
            # Get all candidate tokens (excluding current token)
            candidate_tokens = [token for token in top_k_tokens if token != current_token]
            
            if not candidate_tokens:
                continue
            
            # Convert candidate tokens to token IDs
            candidate_token_ids = [tokenizer.convert_tokens_to_ids(token) for token in candidate_tokens]
            candidate_token_ids = torch.tensor(candidate_token_ids)
            
            # Get token IDs for other classes (unchanged)
            other_class_token_ids = []
            for c in classes:
                if c != class_to_change:
                    other_class_token_ids.append(tokenizer.convert_tokens_to_ids(token_assignments[c]))
            other_class_token_ids = torch.tensor(other_class_token_ids)
            
            # Vectorized calculation for all candidates
            total_log_probs = torch.zeros(len(candidate_tokens))
            
            # for sentence, label in zip(sentences, labels):
            #     sentence_logits_tensor = sentence_logits[sentence]
                
            #     if label == class_to_change:
            #         # This sentence uses the class we're changing
            #         label_logits = sentence_logits_tensor[candidate_token_ids].float()
            #         other_logits = sentence_logits_tensor[other_class_token_ids].float()
                    
            #         # For each candidate, calculate denominator and probability
            #         for i, candidate_logit in enumerate(label_logits):
            #             denominator = torch.exp(candidate_logit) + torch.exp(other_logits).sum()
            #             prob = torch.exp(candidate_logit) / denominator
            #             total_log_probs[i] += torch.log(prob)
            #     else:
            #         # This sentence uses a different class (unchanged)
            #         label_token_id = tokenizer.convert_tokens_to_ids(token_assignments[label])
            #         label_logit = sentence_logits_tensor[label_token_id].float()
                    
            #         # Calculate denominator for each candidate
            #         for i, candidate_logit in enumerate(sentence_logits_tensor[candidate_token_ids].float()):
            #             other_unchanged_logits = sentence_logits_tensor[other_class_token_ids].float()
            #             denominator = torch.exp(candidate_logit) + torch.exp(other_unchanged_logits).sum()
            #             prob = torch.exp(label_logit) / denominator
            #             total_log_probs[i] += torch.log(prob)
            
            # Vectorized calculation for all candidates
            for sentence, label in zip(sentences, labels):
                sentence_logits_tensor = sentence_logits[sentence]
                
                if label == class_to_change:
                    # This sentence uses the class we're changing
                    label_logits = sentence_logits_tensor[candidate_token_ids].float()  # Shape: (num_candidates,)
                    other_logits = sentence_logits_tensor[other_class_token_ids].float()  # Shape: (num_other_classes,)
                    
                    # Vectorized denominator calculation
                    # label_logits.unsqueeze(1) + other_logits.sum() gives us all denominators at once
                    denominators = torch.exp(label_logits) + torch.exp(other_logits).sum()  # Shape: (num_candidates,)
                    probs = torch.exp(label_logits) / denominators
                    total_log_probs += torch.log(probs)
                    
                else:
                    # This sentence uses a different class (unchanged)
                    label_token_id = tokenizer.convert_tokens_to_ids(token_assignments[label])
                    label_logit = sentence_logits_tensor[label_token_id].float()
                    
                    # Vectorized denominator calculation
                    candidate_logits = sentence_logits_tensor[candidate_token_ids].float()  # Shape: (num_candidates,)
                    other_unchanged_logits = sentence_logits_tensor[other_class_token_ids].float()  # Shape: (num_other_classes,)
                    
                    denominators = torch.exp(candidate_logits) + torch.exp(other_unchanged_logits).sum()
                    probs = torch.exp(label_logit) / denominators  # Same numerator for all candidates
                    total_log_probs += torch.log(probs)
            
            # Find the best candidate
            best_idx = torch.argmax(total_log_probs).item()
            best_objective = total_log_probs[best_idx].item()
            best_token = candidate_tokens[best_idx]
            
            # If improvement found, update
            if best_objective > current_objective:
                token_assignments[class_to_change] = best_token
                current_objective = best_objective
                improved = True
                print(f"Iteration {iteration}: Improved objective to {best_objective} by changing {class_to_change} to {best_token}")
                break  # Move to next class after finding improvement
    
    print(f"Final objective after {iteration} iterations: {current_objective}")
    print(f"Final token assignments: {token_assignments}")

    return token_assignments, current_objective

In [ ]:
def optimize_tokens_with_comments(top_k_tokens, sentences, labels, sentence_logits, max_iterations=100):
    classes = labels.unique()
    
    # Convert to list if it's dict_keys
    if hasattr(top_k_tokens, 'keys'):
        top_k_tokens = list(top_k_tokens)
    
    # Helper function to calculate the objective (sum_support_ex)
    def calculate_objective(token_assignments):
        total_log_prob = 0
        for sentence, label in zip(sentences, labels):
            # Convert token string to token ID
            label_token_id = tokenizer.convert_tokens_to_ids(token_assignments[label])
            logit = sentence_logits[sentence][label_token_id]
            
            # Convert to float32 to avoid BFloat16 issues
            if hasattr(logit, 'float'):
                logit = logit.float()
            
            # Calculate denominator
            denominator = 0
            for c in classes:
                c_token_id = tokenizer.convert_tokens_to_ids(token_assignments[c])
                c_logit = sentence_logits[sentence][c_token_id]
                
                # Convert to float32 to avoid BFloat16 issues
                if hasattr(c_logit, 'float'):
                    c_logit = c_logit.float()
                
                denominator += torch.exp(c_logit)
            
            probs = torch.exp(logit) / denominator
            log_prob = torch.log(probs)
            total_log_prob += log_prob.item()  # Use .item() to extract scalar value
        return total_log_prob
    
    # Initialize with random assignment
    token_assignments = {c: random.choice(top_k_tokens) for c in classes}
    current_objective = calculate_objective(token_assignments)
    
    print(f"Initial objective: {current_objective}")
    
    # Hill climbing optimization
    improved = True
    iteration = 0
    
    while improved and iteration < max_iterations:
        improved = False
        iteration += 1
        
        # # Try changing one token assignment at a time
        # for class_to_change in classes:
        #     current_token = token_assignments[class_to_change]
            
        #     # Try all other possible tokens for this class
        #     for new_token in top_k_tokens:
        #         if new_token == current_token:
        #             continue
                    
        #         # Create new assignment with this token change
        #         new_assignments = token_assignments.copy()
        #         new_assignments[class_to_change] = new_token
                
        #         # Calculate new objective
        #         new_objective = calculate_objective(new_assignments)
                
        #         # If improvement found, update and mark as improved
        #         if new_objective > current_objective:
        #             token_assignments = new_assignments
        #             current_objective = new_objective
        #             improved = True
        #             print(f"Iteration {iteration}: Improved objective to {current_objective} by changing {class_to_change} to {new_token}")
        #             break  # Move to next class after finding improvement
            
        #     if improved:
        #         break  # Restart from first class after any improvement
        # Try changing one token assignment at a time
        for class_to_change in classes:
            current_token = token_assignments[class_to_change]
            
            # Get all candidate tokens (excluding current token)
            candidate_tokens = [token for token in top_k_tokens if token != current_token]
            
            if not candidate_tokens:
                continue
            
            # Convert candidate tokens to token IDs
            candidate_token_ids = [tokenizer.convert_tokens_to_ids(token) for token in candidate_tokens]
            candidate_token_ids = torch.tensor(candidate_token_ids)
            
            # Get token IDs for other classes (unchanged)
            other_class_token_ids = []
            for c in classes:
                if c != class_to_change:
                    other_class_token_ids.append(tokenizer.convert_tokens_to_ids(token_assignments[c]))
            other_class_token_ids = torch.tensor(other_class_token_ids)
            
            # Vectorized calculation for all candidates
            total_log_probs = torch.zeros(len(candidate_tokens))
            
            # for sentence, label in zip(sentences, labels):
            #     sentence_logits_tensor = sentence_logits[sentence]
                
            #     if label == class_to_change:
            #         # This sentence uses the class we're changing
            #         label_logits = sentence_logits_tensor[candidate_token_ids].float()
            #         other_logits = sentence_logits_tensor[other_class_token_ids].float()
                    
            #         # For each candidate, calculate denominator and probability
            #         for i, candidate_logit in enumerate(label_logits):
            #             denominator = torch.exp(candidate_logit) + torch.exp(other_logits).sum()
            #             prob = torch.exp(candidate_logit) / denominator
            #             total_log_probs[i] += torch.log(prob)
            #     else:
            #         # This sentence uses a different class (unchanged)
            #         label_token_id = tokenizer.convert_tokens_to_ids(token_assignments[label])
            #         label_logit = sentence_logits_tensor[label_token_id].float()
                    
            #         # Calculate denominator for each candidate
            #         for i, candidate_logit in enumerate(sentence_logits_tensor[candidate_token_ids].float()):
            #             other_unchanged_logits = sentence_logits_tensor[other_class_token_ids].float()
            #             denominator = torch.exp(candidate_logit) + torch.exp(other_unchanged_logits).sum()
            #             prob = torch.exp(label_logit) / denominator
            #             total_log_probs[i] += torch.log(prob)
            
            # Vectorized calculation for all candidates
            for sentence, label in zip(sentences, labels):
                sentence_logits_tensor = sentence_logits[sentence]
                
                if label == class_to_change:
                    # This sentence uses the class we're changing
                    label_logits = sentence_logits_tensor[candidate_token_ids].float()  # Shape: (num_candidates,)
                    other_logits = sentence_logits_tensor[other_class_token_ids].float()  # Shape: (num_other_classes,)
                    
                    # Vectorized denominator calculation
                    # label_logits.unsqueeze(1) + other_logits.sum() gives us all denominators at once
                    denominators = torch.exp(label_logits) + torch.exp(other_logits).sum()  # Shape: (num_candidates,)
                    probs = torch.exp(label_logits) / denominators
                    total_log_probs += torch.log(probs)
                    
                else:
                    # This sentence uses a different class (unchanged)
                    label_token_id = tokenizer.convert_tokens_to_ids(token_assignments[label])
                    label_logit = sentence_logits_tensor[label_token_id].float()
                    
                    # Vectorized denominator calculation
                    candidate_logits = sentence_logits_tensor[candidate_token_ids].float()  # Shape: (num_candidates,)
                    other_unchanged_logits = sentence_logits_tensor[other_class_token_ids].float()  # Shape: (num_other_classes,)
                    
                    denominators = torch.exp(candidate_logits) + torch.exp(other_unchanged_logits).sum()
                    probs = torch.exp(label_logit) / denominators  # Same numerator for all candidates
                    total_log_probs += torch.log(probs)
            
            # Find the best candidate
            best_idx = torch.argmax(total_log_probs).item()
            best_objective = total_log_probs[best_idx].item()
            best_token = candidate_tokens[best_idx]
            
            # If improvement found, update
            if best_objective > current_objective:
                token_assignments[class_to_change] = best_token
                current_objective = best_objective
                improved = True
                print(f"Iteration {iteration}: Improved objective to {best_objective} by changing {class_to_change} to {best_token}")
                break  # Move to next class after finding improvement
    
    print(f"Final objective after {iteration} iterations: {current_objective}")
    print(f"Final token assignments: {token_assignments}")

    return token_assignments, current_objective

In [25]:
print(all_tokens_str)

dict_keys(['.expression', 'sville', 'OutOfRangeException', 'ozÃŃ', 'ĠÐºÐ¾ÑĤÐ¾ÑĢÐ¾Ð¹', 'moil', '_multiple', '?",Ċ', 'sum', 'ĠÐ¿ÑĢÐ¾ÑĨÐµÑģ', 'Ġue', 'ç«¥', 'ĠInitializes', '--;Ċ', 'prot', 'Ġecs', 'reviews', 'Ġconnectors', 'Collapse', 'ëŁ¬ìĬ¤', 'Ġmop', 'Ġì¦', 'ĠÐ¼Ð°Ð»ÑĮ', '_variation', 'ĠØ§ÙħØ±ÙĪØ²', '.Timer', 'Morning', 'massage', 'ulos', '$c', '/pi', 'ĠSonra', 'ĠäºĮ', '&quot', ')$/', 'ain', 'Ġborrowers', 'ĠMultiple', 'ö', 'ĠæĽ', "(_('", 'fadeIn', ');čččĊ', 'ĉtag', 'Cap', 'Ġcarga', 'Ġcampaign', 'ĠsÃ¼reÃ§', 'Posting', 'iltro', 'Ġreforms', 'ĠÑģÐ¿Ð¾Ñģ', 'ĠEconomics', '<|reserved_special_token_107|>', 'Ġinstall', 'ĠhÃ¶', 'soever', 'kf', 'openssl', '-place', 'ĉoption', 'ĠMandela', 'onDelete', 'à¸±à¸Ľ', '-end', '_FILTER', 'Û²Û°', 'Ġ{[', 'Tuy', 'ighbours', 'Commission', 'gae', 'ĠAES', 'Ù¾', 'Ġopenness', 'Ä±yÄ±', 'Ġrecibir', 'á»Ĺ', 'ontology', '.pa', 'à¹Įà¸Ĭ', 'Ġtoured', 'ĠudÃ¡l', 'ĠdataTable', '.define', 'èĨľ', '_Api', 'Ġ------------------------------------------------', 'Ġpos', 'æĪ¸', '.centerX

In [16]:
best_assignments, best_score = optimize_tokens(list(all_tokens_str), sentences, labels, sentence_logits)

Initial objective: -29.497560891175272
Iteration 1: Improved objective to -15.458715438842773 by changing D to quest
Iteration 2: Improved objective to -12.096742630004883 by changing A to Ġcelebr
Iteration 3: Improved objective to -11.487228393554688 by changing D to Event
Iteration 4: Improved objective to -11.253730773925781 by changing A to ad
Iteration 5: Improved objective to -4.809465408325195 by changing C to ile
Iteration 6: Improved objective to -4.637887477874756 by changing D to oot
Iteration 7: Improved objective to -4.460453033447266 by changing A to pt
Final objective after 8 iterations: -4.460453033447266
Final token assignments: {'D': 'oot', 'A': 'pt', 'C': 'ile'}


In [22]:
def W_ICL(top_k_tokens, sentences, labels, sentence_probs):
    W = {}
    for class_label in labels.unique(): # C 
        W[class_label] = {}
        for i, token in enumerate(all_tokens_str): # V  
            W[class_label][token] = 0

    not_zero = 0
    total = 0
    for class_label in labels.unique(): # C
        for i, token in enumerate(all_tokens_str): # V
            sum = 0
            for sentence, label in zip(sentences, labels): # N
                prob = sentence_probs[sentence][i]
                log_prob = torch.log(prob)
                # one_minus_log_prob = torch.log(1 - prob)
                one_minus_log_prob = -torch.log(prob)
                if label == class_label:
                    if token == 'Ġglaciers':    
                        print(f"Log probability of {token} in {sentence}: {log_prob}")
                    sum += log_prob
                else:
                    sum += one_minus_log_prob
                    total += 1
                    if token == 'Ġglaciers':
                        print(f"Log probability of not {token} in {sentence}: {one_minus_log_prob}")
                    if one_minus_log_prob != 0:
                        not_zero += 1
            W[class_label][token] = sum.item()
    print("not_zero: ", not_zero, "total: ", total)

    return W

In [23]:
W = W_ICL(all_tokens_str, sentences, labels, sentence_probs)

Log probability of Ġglaciers in As the riots intensified, I cowered in my basement, wondering if democracy had finally met its Waterloo.: -22.875
Log probability of not Ġglaciers in The concert was a tidal wave of sound, washing over the ecstatic crowd.: 22.125
Log probability of not Ġglaciers in The opposition's venomous rhetoric spews forth, poisoning the well of civil discourse.: 22.125
Log probability of not Ġglaciers in he's thrilled to bits about his new diet of nothing but deep-fried foods and sugary drinks.: 19.625
Log probability of not Ġglaciers in The corrupt senator's lies finally caught up with him when the incriminating evidence surfaced.: 20.875
Log probability of not Ġglaciers in The Wi-Fi connection was so blazingly fast, I managed to load half a webpage in just under an hour.: 22.0
Log probability of not Ġglaciers in In the coming election, I'll cast my ballot and watch it soar like a dove, carrying the weight of millions of dreams.: 23.0
Log probability of Ġglaciers 

In [24]:
import numpy as np

In [25]:
def reassign_labels(all_tokens_str, labels, sentences, sentence_probs, tokenizer):
    num_to_label = {}
    num_to_class = {}
    W = W_ICL(all_tokens_str, sentences, labels, sentence_probs)
    weights = []
    
    for (i, v1) in enumerate(W):
        num_to_class[i] = v1
        for (j, v2) in enumerate(W[v1]):
            num_to_label[j] = v2

    weights = []
    for (i, v1) in enumerate(W):
        weights.append([])
        for (j, v2) in enumerate(W[v1]):
            weights[i].append(W[v1][v2])
    cost = np.array(weights)

    row_ind, col_ind = linear_sum_assignment(cost, maximize=True)

    new_labels = {}
    for i in range(len(row_ind)):
        token_str = num_to_label[col_ind[i]]
        token_id = tokenizer.convert_tokens_to_ids(token_str)
        
        print(f"Class: {num_to_class[row_ind[i]]}")
        print(f"  Token string: '{token_str}'")
        print(f"  Token ID: {token_id}")
        print(f"  Back to string: '{tokenizer.convert_ids_to_tokens(token_id)}'")
        
        new_labels[num_to_class[row_ind[i]]] = (token_str, token_id)
    print("new_labels: ", new_labels)
    print("sum of cost: ", cost[row_ind, col_ind].sum())
    return new_labels

In [26]:
reassign_labels(all_tokens_str, labels, sentences, sentence_probs, tokenizer)

Log probability of Ġglaciers in As the riots intensified, I cowered in my basement, wondering if democracy had finally met its Waterloo.: -22.875
Log probability of not Ġglaciers in The concert was a tidal wave of sound, washing over the ecstatic crowd.: 22.125
Log probability of not Ġglaciers in The opposition's venomous rhetoric spews forth, poisoning the well of civil discourse.: 22.125
Log probability of not Ġglaciers in he's thrilled to bits about his new diet of nothing but deep-fried foods and sugary drinks.: 19.625
Log probability of not Ġglaciers in The corrupt senator's lies finally caught up with him when the incriminating evidence surfaced.: 20.875
Log probability of not Ġglaciers in The Wi-Fi connection was so blazingly fast, I managed to load half a webpage in just under an hour.: 22.0
Log probability of not Ġglaciers in In the coming election, I'll cast my ballot and watch it soar like a dove, carrying the weight of millions of dreams.: 23.0
Log probability of Ġglaciers 

{'D': ('elm', 24037), 'A': ('InputStream', 10689), 'C': ('panse', 95519)}